***Welcome! Hello!***

We are glad you're here 👏! And are excited to get you scanning 🔍!

This notebook exists to enable you to perform UDS (Unified Diagnostic Services) enumeration on an ECU.

# How To Use

It does this with `python` BUT don't worry: you don't need to know `python` to use this notebook.

Start by running the cells.

Run all cells in both *1. Basic Imports* and *2. Scapy Setup* sections below:
1. click the cell
2. click the run button (or shift+enter)

Then move to the *Do the thing: UDS Scanning* section and run those cells too 🚀!

(For the lazy / impatient: you can also run all the cells with the ⏩ button on the toolbar).

## 1. Basic Imports

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact_manual

import platform
from tabulate import tabulate
import logging

from scapy.all import load_layer, conf, load_contrib, Raw, raw

load_layer("can")
conf.contribs["CANSocket"] = {"use-python-can": True}
load_contrib("cansocket")
load_contrib("isotp")
load_contrib("automotive.uds")
load_contrib("automotive.uds_scan")

from scapy.contrib.cansocket import CANSocket
from scapy.contrib.isotp import ISOTPSocket
from scapy.contrib.automotive.uds import *
from scapy.contrib.automotive.uds_scan import (
    UDS_Scanner,
    UDS_ServiceEnumerator,
    UDS_RDBIEnumerator,
    UDS_RDBISelectiveEnumerator,
    UDS_DSCEnumerator,
    UDS_SAEnumerator,
)
from scapy.contrib.automotive import log_automotive

IS_WINDOWS = platform.system() == "Windows"
DEFAULT_INTERFACE = "candle" if IS_WINDOWS else "socketcan"
DEFAULT_CHANNEL = "0" if IS_WINDOWS else "can0"

## 2. `scapy` Setup

We've done the basic imports. Now we need to setup `scapy` automotive CAN stuff.

In [ ]:
# Set logging to DEBUG to see full activity during scans
log_automotive.setLevel(logging.INFO)
logging.getLogger("scapy.contrib.automotive.uds_scan").setLevel(logging.DEBUG)

CURRENT_CSOCK = None
CURRENT_ISOCK = None


def socket_factory_factory(interface, channel, bitrate, send_to_id, recv_from_id):
    def factory():
        global CURRENT_CSOCK, CURRENT_ISOCK
        if CURRENT_ISOCK:
            try:
                CURRENT_ISOCK.close()
            except Exception:
                pass
        if CURRENT_CSOCK:
            try:
                CURRENT_CSOCK.close()
            except Exception:
                pass

        is_extended = recv_from_id > 0x7FF
        mask = 0x1FFFFFFF if is_extended else 0x7FF
        can_filters = [
            {"can_id": recv_from_id, "can_mask": mask, "extended": is_extended}
        ]

        CURRENT_CSOCK = CANSocket(
            interface=interface,
            channel=channel,
            bitrate=bitrate,
            can_filters=can_filters,
            sleep_after_open=0.0001,
        )
        CURRENT_ISOCK = ISOTPSocket(
            CURRENT_CSOCK,
            tx_id=send_to_id,
            rx_id=recv_from_id,
            padding=True,
            basecls=UDS,
        )
        return CURRENT_ISOCK

    return factory

# Do the thing: UDS Scanning

The following sections allow you to scan for different UDS features. 
Adjust the connection settings (interface, channel, bitrate, IDs) in each section as needed.

In [ ]:
# Common UDS Service Requests (Complete Packets)
COMMON_SIDs = [
    0x10,  # Diagnostic Session Control (defaultSession)
    0x11,  # ECU Reset (hardReset)
    0x14,  # Clear Diagnostic Information
    0x19,  # Read DTC Information
    0x22,  # Read Data By Identifier (VIN)
    0x27,  # Security Access
    0x28,  # Communication Control
    0x3E,  # Tester Present
    0x85,  # Control DTC Setting
]

# Common UDS Data Identifiers (DIDs)
COMMON_DIDs = list(range(0xF180, 0xF1A0))  # Standardized range


def print_summary_table(enumerator, show_only_found=False):
    summary_data = []
    for state, req, resp, start, end in enumerator.results:
        sid = req.service
        sid_hex = hex(sid)
        sid_name = req.get_field("service").i2s.get(sid, "Unknown")
        if sid_name == "Unknown":
            obd_names = {
                0x01: "OBDII Mode 1 (Current Data)",
                0x02: "OBDII Mode 2 (Freeze Frame Data)",
                0x03: "OBDII Mode 3 (Stored DTCs)",
                0x04: "OBDII Mode 4 (Clear DTCs)",
                0x05: "OBDII Mode 5 (Oxygen Sensor Monitoring)",
                0x06: "OBDII Mode 6 (System Monitoring)",
                0x07: "OBDII Mode 7 (Pending DTCs)",
                0x08: "OBDII Mode 8 (Control On-Board Component)",
                0x09: "OBDII Mode 9 (Vehicle Information)",
                0x0A: "OBDII Mode A (Permanent DTCs)",
            }
            sid_name = obd_names.get(sid, "Unknown")
        left_col = f"{sid_hex} ({sid_name})"

        if resp:
            # NRC 0x11 (serviceNotSupported) and 0x7F (serviceNotSupportedInActiveSession)
            # are treated as a failure to find the service
            is_nrc = getattr(resp, "service", None) == 0x7F
            nrc = getattr(resp, "negativeResponseCode", None) if is_nrc else None
            if is_nrc and nrc in [0x11, 0x7F]:
                status = "❌"
            else:
                status = "✅"
            resp_str = repr(resp)
        else:
            status = "❌"
            resp_str = "Timeout"

        if show_only_found and status == "❌":
            continue

        summary_data.append([left_col, f"{status} {resp_str}"])

    print("\nSummary Table:")
    print(
        tabulate(
            summary_data,
            headers=["Service", "Response"],
            tablefmt="fancy_grid",
            maxcolwidths=[30, 80],
        )
    )


def print_did_summary_table(enumerator, show_only_found=False):
    from scapy.all import hexdump
    from scapy.contrib.automotive.bmw.definitions import UDS_RDBI

    all_results = []
    if hasattr(enumerator, "results"):
        all_results = enumerator.results
    elif hasattr(enumerator, "test_cases"):
        for tc in enumerator.test_cases:
            if hasattr(tc, "results"):
                all_results.extend(tc.results)

    # De-duplicate results by DID, prioritizing positive responses
    results_dict = {}
    for state, req, resp, start, end in all_results:
        did = (
            req.identifiers[0]
            if hasattr(req, "identifiers") and req.identifiers
            else None
        )
        if did is not None:
            is_positive = resp and resp.service != 0x7F
            if did not in results_dict or is_positive:
                results_dict[did] = (state, req, resp, start, end)

    summary_data = []
    for did in sorted(results_dict.keys()):
        state, req, resp, start, end = results_dict[did]

        is_nrc = getattr(resp, "service", None) == 0x7F
        status = "✅" if resp and not is_nrc else "❌"

        if show_only_found and status == "❌":
            continue

        did_name = ""
        if 0xF180 <= did <= 0xF1FF:
            did_name = UDS_RDBI.dataIdentifiers.get(did, "")
        name_str = f" ({did_name})" if did_name else ""
        left_col = f"{did} ({hex(did)}){name_str}"

        if resp:
            resp_decode = repr(resp)

            if status == "✅":
                # Omit service ID (1 byte) and DID (2 bytes) for positive RDBI responses
                payload = raw(resp)[3:] if resp.service == 0x62 else raw(resp)
                payload_hex = hexdump(payload, dump=True) if payload else ""
                right_col = f"{status} {resp_decode}\n{payload_hex}"
            else:
                right_col = f"{status} {resp_decode}"
        else:
            right_col = "❌ Timeout"

        summary_data.append([left_col, right_col])

    print("\nSummary Table:")
    print(
        tabulate(
            summary_data,
            headers=["DID", "Response"],
            tablefmt="fancy_grid",
            maxcolwidths=[20, 100],
        )
    )

### 1. Enumerating Common Services

This section scans for common UDS services (Diagnostic Session Control, ECU Reset, Security Access, etc.) to see which are supported by the ECU.

In [ ]:
@interact_manual
def scan_common_services(
    interface=widgets.Textarea(DEFAULT_INTERFACE),
    channel=widgets.Textarea(DEFAULT_CHANNEL),
    bitrate=[500000, 250000, 125000, 1000000],
    send_to_id=widgets.Textarea("7e1"),
    recv_from_id=widgets.Textarea("7e9"),
    timeout=widgets.FloatText(1.0),
):
    send_to_id_int = int(send_to_id, 16)
    recv_from_id_int = int(recv_from_id, 16)

    socket_factory = socket_factory_factory(
        interface, channel, bitrate, send_to_id_int, recv_from_id_int
    )

    print(f"Enumerating common UDS services on {send_to_id}...")
    scanner = UDS_Scanner(
        socket_factory(),
        test_cases=[UDS_ServiceEnumerator],
        reconnect_handler=socket_factory,
        UDS_ServiceEnumerator_kwargs={"scan_range": COMMON_SIDs, "timeout": timeout},
    )
    scanner.scan()

    print_summary_table(scanner.configuration.test_cases[0])

### 2. Enumerating Rarer Services

This section scans the entire range of possible UDS Service IDs (0x00 to 0xFF) excluding the common ones, to find any non-standard or rarer services.

**Note: The summary table will only list services that were found (positive responses or NRCs other than 'serviceNotSupported').**

In [ ]:
@interact_manual
def scan_rarer_services(
    interface=widgets.Textarea(DEFAULT_INTERFACE),
    channel=widgets.Textarea(DEFAULT_CHANNEL),
    bitrate=[500000, 250000, 125000, 1000000],
    send_to_id=widgets.Textarea("7e1"),
    recv_from_id=widgets.Textarea("7e9"),
    timeout=widgets.FloatText(0.5),
):
    send_to_id_int = int(send_to_id, 16)
    recv_from_id_int = int(recv_from_id, 16)

    socket_factory = socket_factory_factory(
        interface, channel, bitrate, send_to_id_int, recv_from_id_int
    )

    rare_sids = [sid for sid in range(0x00, 0x100) if sid not in COMMON_SIDs]
    print(
        f"Enumerating {len(rare_sids)} rarer UDS services (0x00-0xFF) "
        f"on {send_to_id}..."
    )
    scanner = UDS_Scanner(
        socket_factory(),
        reconnect_handler=socket_factory,
        test_cases=[UDS_ServiceEnumerator],
        UDS_ServiceEnumerator_kwargs={"scan_range": rare_sids, "timeout": timeout},
    )
    scanner.scan()

    print_summary_table(scanner.configuration.test_cases[0], show_only_found=True)

### 3. Enumerating Common Read DIDs

Data Identifiers (DIDs) are used to read specific data from the ECU (like VIN, software version, etc.). This section scans for commonly used and standardized DIDs.

In [ ]:
@interact_manual
def scan_common_dids(
    interface=widgets.Textarea(DEFAULT_INTERFACE),
    channel=widgets.Textarea(DEFAULT_CHANNEL),
    bitrate=[500000, 250000, 125000, 1000000],
    send_to_id=widgets.Textarea("7e1"),
    recv_from_id=widgets.Textarea("7e9"),
    timeout=widgets.FloatText(1.0),
):
    send_to_id_int = int(send_to_id, 16)
    recv_from_id_int = int(recv_from_id, 16)

    socket_factory = socket_factory_factory(
        interface, channel, bitrate, send_to_id_int, recv_from_id_int
    )

    print(f"Enumerating common Read DIDs on {send_to_id}...")
    scanner = UDS_Scanner(
        socket_factory(),
        reconnect_handler=socket_factory,
        test_cases=[UDS_RDBIEnumerator],
        UDS_RDBIEnumerator_kwargs={"scan_range": COMMON_DIDs, "timeout": timeout},
    )
    scanner.scan()

    print_did_summary_table(scanner.configuration.test_cases[0])

### 4. Enumerating Remaining DIDs

This section uses the `UDS_RDBISelectiveEnumerator` to efficiently scan for all supported DIDs supported by the ECU.

**Note: The summary table will only list DIDs that were successfully found.**

NOTE: this takes some time (to scan a large space) and it prints a bunch of 'red' text as progress updates.

In [ ]:
@interact_manual
def scan_remaining_dids(
    interface=widgets.Textarea(DEFAULT_INTERFACE),
    channel=widgets.Textarea(DEFAULT_CHANNEL),
    bitrate=[500000, 250000, 125000, 1000000],
    send_to_id=widgets.Textarea("7e1"),
    recv_from_id=widgets.Textarea("7e9"),
    timeout=widgets.FloatText(0.1),
):
    send_to_id_int = int(send_to_id, 16)
    recv_from_id_int = int(recv_from_id, 16)

    socket_factory = socket_factory_factory(
        interface, channel, bitrate, send_to_id_int, recv_from_id_int
    )

    print(f"Enumerating all supported Read DIDs on {send_to_id} (Selective)...")
    scanner = UDS_Scanner(
        socket_factory(),
        reconnect_handler=socket_factory,
        test_cases=[UDS_RDBISelectiveEnumerator],
        UDS_RDBISelectiveEnumerator_kwargs={
            "timeout": timeout,
        },
    )
    scanner.scan()

    print_did_summary_table(scanner.configuration.test_cases[0], show_only_found=True)

### 5. Enumerating All DIDs (Brute Force)

This section scans the entire possible range of DIDs (0x0000 to 0xFFFF). 

**WARNING: This will take a very, very long time (hours depending on your connection and ECU).**

Note: The summary table will only list DIDs that were successfully found.

In [ ]:
@interact_manual
def scan_all_dids_bruteforce(
    interface=widgets.Textarea(DEFAULT_INTERFACE),
    channel=widgets.Textarea(DEFAULT_CHANNEL),
    bitrate=[500000, 250000, 125000, 1000000],
    send_to_id=widgets.Textarea("7e1"),
    recv_from_id=widgets.Textarea("7e9"),
    timeout=widgets.FloatText(0.05),
    inter=widgets.FloatText(0.01),
):
    send_to_id_int = int(send_to_id, 16)
    recv_from_id_int = int(recv_from_id, 16)

    socket_factory = socket_factory_factory(
        interface, channel, bitrate, send_to_id_int, recv_from_id_int
    )

    print(f"Brute-forcing all 65536 DIDs on {send_to_id}...")
    scanner = UDS_Scanner(
        socket_factory(),
        reconnect_handler=socket_factory,
        test_cases=[UDS_RDBIEnumerator],
        UDS_RDBIEnumerator_kwargs={
            "scan_range": range(0x10000),
            "timeout": timeout,
            "inter": inter,
        },
    )
    scanner.scan()

    print_did_summary_table(scanner.configuration.test_cases[0], show_only_found=True)

### 6. Enumerating DSC and SA Levels

This section scans for available Diagnostic Sessions (DSC) and Security Access (SA) levels that can be requested from the ECU without prior authentication.

NOTE: takes some time and will print a bunch of 'red' text for progress updates.

In [ ]:
@interact_manual
def scan_dsc_sa(
    interface=widgets.Textarea(DEFAULT_INTERFACE),
    channel=widgets.Textarea(DEFAULT_CHANNEL),
    bitrate=[500000, 250000, 125000, 1000000],
    send_to_id=widgets.Textarea("7e1"),
    recv_from_id=widgets.Textarea("7e9"),
    timeout=widgets.FloatText(1.0),
):
    send_to_id_int = int(send_to_id, 16)
    recv_from_id_int = int(recv_from_id, 16)

    socket_factory = socket_factory_factory(
        interface, channel, bitrate, send_to_id_int, recv_from_id_int
    )

    print(f"Enumerating DSC and SA levels on {send_to_id}...")
    scanner = UDS_Scanner(
        socket_factory(),
        reconnect_handler=socket_factory,
        test_cases=[UDS_DSCEnumerator, UDS_SAEnumerator],
        UDS_DSCEnumerator_kwargs={"timeout": timeout},
        UDS_SAEnumerator_kwargs={"timeout": timeout},
    )
    scanner.scan()

    scanner.show_testcases_status()
    scanner.show_testcases()